# Milestone 2: RAG Pipeline Exploration and Evaluation

## 1. Setup and Imports

Set up project paths, import retrieval utilities, and check that required artifacts exist.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed_data_sample.parquet"
SEMANTIC_STORE_PATH = PROJECT_ROOT / "data" / "processed" / "faiss_store"
BM25_PATH = PROJECT_ROOT / "data" / "processed" / "bm25_retriever.pkl"

from src.semantic import get_or_build_vectorstore, get_semantic_retriever
from src.bm25 import load_bm25_retriever

assert DATA_PATH.exists(), f"Missing parquet file: {DATA_PATH}"
assert SEMANTIC_STORE_PATH.exists(), f"Missing semantic store: {SEMANTIC_STORE_PATH}"

print("Setup complete.")

/Users/clairesaunders/miniforge3/envs/retrieval-app/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


## 2. Load Retrieval Components 

Load the saved semantic vector store and BM25 retriever.

In [2]:
semantic_vectorstore = get_or_build_vectorstore(
    data_path=str(DATA_PATH),
    store_path=str(SEMANTIC_STORE_PATH),
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

bm25 = load_bm25_retriever(str(BM25_PATH))

print("BM25 retriever loaded." if bm25 is not None else "BM25 retriever not found.")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 10357.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded semantic vector store.
BM25 retriever loaded.


## 3. Semantic Retrieval Sanity Checks

Create a semantic retriever and inspect a few retrieved documents to confirm that retrieval is working as expected. 

In [3]:
semantic_retriever = get_semantic_retriever(semantic_vectorstore, top_k=5)
print("Semantic retriever ready.")

Semantic retriever ready.


In [4]:
# Test Semantic Retriever
query = "good gift for someone who loves gardening"
docs = semantic_retriever.invoke(query)

print(f"Retrieved {len(docs)} documents for test query.")

Retrieved 5 documents for test query.


In [5]:
test_queries = [
    "blue patio chair cushion",
    "good gift for someone who loves gardening",
    "something to make a garden look more colorful and lively"
]

for q in test_queries:
    retrieved_docs = semantic_retriever.invoke(q)
    print(f"\nQUERY: {q}")
    for i, doc in enumerate(retrieved_docs[:2], start=1):
        print(f"\nResult {i}")
        print("Title:", doc.metadata.get("product_title"))
        print("Review:", doc.metadata.get("review_text"))
        print("Rating:", doc.metadata.get("rating"))


QUERY: blue patio chair cushion

Result 1
Title: Pillow Perfect Stripe Indoor/Outdoor 1 Piece Split Back Round Corner Chair Seat Cushion with Ties, Deep Seat, Weather, and Fade Resistant, 40.5" x 21", Red/White Midland, 1 Count
Review: I must have the oddest-sized patio chairs ever made because I could not find replacement cushions for them. I finally looked on Amazon and found these. They are still just a tad bit too big, but they work and were a decent price, as well. I was also trying to match my existing umbrella. I am really pleased with my purchase.
Rating: 5

Result 2
Title: Rocking Chair Cushion Pad, 47IN Indoor Outdoor Navy Blue Rocking Chair Cushions Set Soft Thickened Patio Chaise Lounger Cushion Overstuffed Patio Chair Cushion
Review: Very flimsy, no comfort at all, purchased navy blue which was very pretty but within one month it faded to grey. Very disappointing.
Rating: 1

QUERY: good gift for someone who loves gardening

Result 1
Title: Funny Combat Garden Gnome Riding

The retrieved results appeared relevant across both keyword-heavy and more semantic queries, suggesting that the FAISS-based semantic retriever was functioning as intended.

## 4. Context Building

Format retrieved documents into a structured context block for the LLM.

In [6]:
def build_context(docs):
    """
    Convert retrieved LangChain documents into a structured context block.
    """
    context_blocks = []

    for i, doc in enumerate(docs, start=1):
        price = doc.metadata.get("price", "N/A")
        if price in [None, "", "N/A", "nan", "NaN"]:
            price_line = ""
        else:
            price_line = f"- Price: {price}\n"

        review_text = str(doc.metadata.get("review_text", "N/A"))[:400]
        features = str(doc.metadata.get("features", "N/A"))[:250]

        block = (
            f"Document {i}\n"
            f"Product title: {doc.metadata.get('product_title', 'N/A')}\n"
            f"Review title: {doc.metadata.get('review_title', 'N/A')}\n"
            f"Review text: {review_text}\n"
            f"Review evidence:\n"
            f"- Review rating: {doc.metadata.get('rating', 'N/A')}\n"
            f"- Average product rating: {doc.metadata.get('average_rating', 'N/A')}\n"
            f"- Number of ratings: {doc.metadata.get('rating_number', 'N/A')}\n"
            f"{price_line}"
            f"- Features: {features}\n"
        )
        context_blocks.append(block)

    return "\n\n---\n\n".join(context_blocks)

In [7]:
context = build_context(docs)
print(context[:2000])

Document 1
Product title: Funny Combat Garden Gnome Riding a Flamingo – Cute Home Decor Statue, for Lawn, Yard, or Office - Indoor Outdoor 11 Inch Tall, by Dodo Creations
Review title: Perfect
Review text: It is a nice size
Review evidence:
- Review rating: 5
- Average product rating: 4.0
- Number of ratings: 76
- Features: 💗 ARMED WITH CUTENESS -Atop a military pink flamingo, our gnome is an expert in gardens security. 💗 FUNNY DECOR -Our decorative figures make an eye-catching ornament. Perfect on your desk or shelf. 💗 DURABLE & WEATHERPROOF - Poly-resin statues able t


---

Document 2
Product title: Garden Set Birthday Gifts for Mom. Wearable Garden Tool Set with Knee Pad & Garden Tools. Ideal Grandma Gifts & Mom Gifts. Gift Box Includes Garden Caddy, Tools & Kneeling Mat Birthday Gifts (Green)
Review title: Good Product
Review text: Gift for Daughter
Review evidence:
- Review rating: 5
- Average product rating: 4.5
- Number of ratings: 65
- Price: 59.95
- Features: GIFT SET FOR WOM

Including review text together with rating, average rating, and number of ratings helped the model use both qualitative and quantitative evidence when making recommendations. During hybrid RAG testing, including all metadata fields made the prompt too large for the LLM token limit, so the final context builder removed categories and descriptions and kept only the most useful fields for grounded recommendations.

## 5. LLM Setup

Use a hosted Groq model for fast iterative experimentation without requiring local GPU resources.

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

print("API key loaded:", os.getenv("GROQ_API_KEY") is not None)

API key loaded: True


In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

response = llm.invoke("In one sentence, what makes a product review useful?")

print(response.content)

A product review is considered useful when it provides detailed, honest, and balanced feedback from a real user, highlighting both the product's strengths and weaknesses, to help others make informed purchasing decisions.


We selected `llama-3.1-8b-instant` via Groq because it provides a strong balance of response quality and very fast inference, making it well-suited for iterative RAG development on a local machine without a GPU.

## 6. Prompt Template Experiments

Test three prompt styles: a loose baseline, a strict grounded prompt, and a balanced prompt designed to support concise but useful recommendations.

In [10]:
SYSTEM_PROMPT_1 = """
You are a helpful Amazon shopping assistant.

Answer the user's question using the provided product reviews and metadata.
Provide helpful suggestions and explain your reasoning.
"""

SYSTEM_PROMPT_2 = """
You are a helpful Amazon shopping assistant.

Answer the user's question using ONLY the provided Amazon product review and metadata context.
Do not make up any information.

If the context is not sufficient to answer confidently, say that the available reviews do not provide enough information.

Keep the answer short, factual, and grounded in the retrieved documents.
"""

SYSTEM_PROMPT_3 = """
You are a helpful Amazon shopping assistant.

Answer the user's question using ONLY the provided Amazon product review and metadata context.
Do not make up details that are not supported by the context.
If the context is not sufficient to answer confidently, say so.

When answering:
- keep the answer concise and practical
- recommend only products supported by the retrieved context
- mention product titles when helpful
- avoid repeating duplicate or near-duplicate products
- always give at least 3 recommendations, and include up to 5 when more good options exist
"""


In [11]:
def build_prompt(query, context, system_prompt):
    """
    Build the final RAG prompt from the system prompt, retrieved context,
    and user query.
    """
    return f"""{system_prompt}

Context:
{context}

Question:
{query}

Answer:"""

## 7. Semantic RAG Prototype

Build and test a simple semantic RAG function before moving the final implementation into `src/rag_pipeline.py`.


In [12]:
def run_semantic_rag(query, retriever, llm, system_prompt):
    """
    Run a semantic RAG pipeline:
    query -> retrieve documents -> build context -> build prompt -> generate answer
    """
    docs = retriever.invoke(query)
    context = build_context(docs)
    prompt = build_prompt(query, context, system_prompt)
    response = llm.invoke(prompt)

    answer = response.content if hasattr(response, "content") else str(response)

    return {
        "query": query,
        "answer": answer,
        "docs": docs,
        "context": context,
        "prompt": prompt,
    }

In [13]:
query = "What would be a good gift for someone who loves gardening?"

for i, system_prompt in enumerate(
    [SYSTEM_PROMPT_1, SYSTEM_PROMPT_2, SYSTEM_PROMPT_3], start=1
):
    result = run_semantic_rag(
        query=query,
        retriever=semantic_retriever,
        llm=llm,
        system_prompt=system_prompt
    )

    print(f"\nPROMPT VARIANT {i}")
    print(result["answer"])
    print("-" * 80)


PROMPT VARIANT 1
Based on the product reviews and metadata provided, here are some gift suggestions for someone who loves gardening:

1. **Garden Tool Set**: A comprehensive set of gardening tools, such as the JUMPHIGH 10 Pieces Gardening Tools Set or the Scuddles Garden Tools Set, would be a thoughtful gift. These sets usually include a variety of tools, such as rakes, shovels, trowels, and pruners, making it easy for the gardener to tackle different tasks.
2. **Personalized Garden Gift**: A customized garden gift, like the Mothers Day Gift Basket, adds a touch of thoughtfulness and uniqueness to the gift. This basket includes a personalized plantable message card, making it a special and meaningful gift.
3. **Decorative Garden Gnome**: A whimsical garden gnome, such as the Funny Combat Garden Gnome Riding a Flamingo, can add a fun and playful touch to the garden. These gnomes often have a unique design and can be a conversation starter.
4. **Gardening Accessories**: Consider gifting

## 8. Final Prompt Choice

Based on the prompt experiments, the final prompt was updated to encourage grounded recommendations while also considering ratings, average rating, and number of ratings.

In [14]:
SYSTEM_PROMPT_FINAL = """
You are a helpful Amazon shopping assistant for Patio, Lawn and Garden products.

Answer the user's question using ONLY the provided Amazon product review and metadata context.
Use professional and friendly language and do not make up details that are not supported by the context.
If the context is not sufficient to answer confidently, say so.

When answering:
- keep the answer concise and practical
- recommend only products supported by the retrieved context
- always give at least 3 recommendations (up to 5 if useful)
- mention product titles
- explain why each product is a good gift in a natural way, using review insights when helpful
- focus on what makes each option appealing (e.g., practical, unique, durable, beginner-friendly)
- do NOT list raw rating statistics unless they are especially important
- do NOT include reviewer names or quote review titles
- highlight differences between options when relevant
- avoid generic phrases like "this is a great gift" without adding a specific reason
- do not include a concluding summary sentence
"""

In [15]:
# Test pipeline 
query = "What would be a good gift for someone who loves gardening?"

result = run_semantic_rag(
    query=query,
    retriever=semantic_retriever,
    llm=llm,
    system_prompt=SYSTEM_PROMPT_FINAL
)

print(result["answer"])

Based on the provided reviews, here are some gift ideas for someone who loves gardening:

1. **Funny Combat Garden Gnome Riding a Flamingo** - This cute and durable garden statue is perfect for adding some personality to their garden or office. It's a unique and fun way to show your appreciation for their love of gardening. ($14.95)
2. **Garden Set Birthday Gifts for Mom** - This gift set comes with a wearable garden tool set, knee pad, and garden caddy, making it a practical and thoughtful gift for any gardener. The gift box and coordinated design make it a lovely present. ($59.95)
3. **Garden Tools Set, JUMPHIGH 10 Pieces Gardening Tools** - This gardening tool set is an excellent gift for any gardener, with 10 essential tools and a carrying case to keep everything organized. It's a great value for the price and comes with a purple floral print design. ($35.99)
4. **Mothers Day Gift Basket - Premium 6 Piece Gardening Presents** - This customizable gift basket is a thoughtful way to s

Prompt variant 1 produced more natural and detailed responses, but often introduced unsupported reasoning. Prompt variant 2 was more grounded but sometimes too restrictive. The final prompt provided the best balance, producing concise, evidence-based recommendations while also encouraging at least 3 grounded suggestions when the context supported them.

## 9. Validate Final Script Implementation

After exploration in the notebook, the final semantic RAG pipeline was moved into `src/rag_pipeline.py` and tested here by importing the script directly.

In [16]:
from src.rag_pipeline import (
    get_semantic_rag_components,
    run_semantic_rag,
    SYSTEM_PROMPT_FINAL,
)

vectorstore, semantic_retriever, llm = get_semantic_rag_components(top_k=5)

result = run_semantic_rag(
    query="What would be a good gift for someone who loves gardening?",
    retriever=semantic_retriever,
    llm=llm,
    system_prompt=SYSTEM_PROMPT_FINAL,
)

print(result["answer"])

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 11304.81it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded semantic vector store.
Based on the provided reviews and metadata, here are some gift ideas for someone who loves gardening:

1. **Funny Combat Garden Gnome Riding a Flamingo** by Dodo Creations - This decorative figure is a unique and humorous addition to any garden or office. Its durable and weatherproof design makes it a practical choice. ($19.99)
2. **Garden Set Birthday Gifts for Mom** by - This wearable garden tool set is a thoughtful gift for any gardener, allowing them to carry their tools comfortably and easily. The set includes a knee pad and garden caddy, making it a comprehensive gift. ($59.95)
3. **Garden Tools Set** by JUMPHIGH - This 10-piece gardening tool set is a great value for its price, offering a variety of tools for different gardening tasks. The set comes with a carrying case, making it easy to store and transport. ($35.99)
4. **Mothers Day Gift Basket - Premium 6 Piece Gardening Presents** - This gift basket is a unique and personalized way to show appre

## 10. Hybrid RAG Exploration

Next, the semantic RAG pipeline was extended by replacing the semantic retriever with a hybrid retriever that combines BM25 keyword retrieval and semantic retrieval. The hybrid pipeline was then tested on the same gardening gift query to compare the style and breadth of the generated recommendations.

In [17]:
from src.hybrid import get_hybrid_retriever
from src.rag_pipeline_hybrid import run_hybrid_rag
from src.semantic import get_semantic_retriever

semantic_retriever_hybrid = get_semantic_retriever(vectorstore, top_k=3)
if hasattr(bm25, "k"):
    bm25.k = 3

hybrid_retriever = get_hybrid_retriever(semantic_retriever_hybrid, bm25)

hybrid_result = run_hybrid_rag(
    query="What would be a good gift for someone who loves gardening?",
    hybrid_retriever=hybrid_retriever,
    llm=llm
)

print(hybrid_result["answer"])

Based on the provided context, here are some gift ideas for someone who loves gardening:

1. **Funny Combat Garden Gnome Riding a Flamingo – Cute Home Decor Statue**: This decorative figure is a unique and fun addition to any garden or office space. Its durable and weatherproof design makes it a great gift that will withstand the elements.

2. **Garden Set Birthday Gifts for Mom. Wearable Garden Tool Set with Knee Pad & Garden Tools**: This practical gift is perfect for any gardener who values convenience and ease. The wearable garden bag and custom kneeler pad make it easy to carry tools and pots around the garden.

3. **Mothers Day Gift Basket - Premium 6 Piece Gardening Presents**: This gift basket is a thoughtful and personalized way to show appreciation for someone's love of gardening. The custom plantable message card and beautiful flowers make it a unique and lasting gift.

4. **UP THE MOMENT Plant Lady Hat**: This fun and functional hat is a great gift for any plant lover. Its 

## Hybrid/Semantic RAG Comparison 

In [18]:
from src.rag_pipeline import (
    get_semantic_rag_components,
    run_semantic_rag,
    SYSTEM_PROMPT_FINAL,
)
from src.semantic import get_semantic_retriever
from src.hybrid import get_hybrid_retriever
from src.rag_pipeline_hybrid import run_hybrid_rag
from src.bm25 import load_bm25_retriever

# Keep semantic RAG at top_k=5
vectorstore, semantic_retriever, llm = get_semantic_rag_components(top_k=5)

# Load BM25
bm25 = load_bm25_retriever("../data/processed/bm25_retriever.pkl")

# Use the same semantic retriever method, but smaller k for hybrid
semantic_retriever_hybrid = get_semantic_retriever(vectorstore, top_k=3)

# Reduce BM25 k too, if supported
if hasattr(bm25, "k"):
    bm25.k = 3

# Hybrid retriever
hybrid_retriever = get_hybrid_retriever(semantic_retriever_hybrid, bm25)

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 10817.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded semantic vector store.


In [19]:
queries_first_half = [
    "blue patio chair cushion",
    "left handed stainless steel garden trowel",
    "6-inch ceramic plant pot with drainage hole",
    "something to make a garden look more colorful and lively",
    "tool for cutting back a large tree",
]

In [20]:
import time

for q in queries_first_half:
    print("=" * 80)
    print(f"QUERY: {q}")
    print("=" * 80)

    semantic_result = run_semantic_rag(
        query=q,
        retriever=semantic_retriever,
        llm=llm,
        system_prompt=SYSTEM_PROMPT_FINAL,
    )

    print("\n--- Semantic RAG ---\n")
    print(semantic_result["answer"])

    time.sleep(1)  # 👈 prevents rate limit

    hybrid_result = run_hybrid_rag(
        query=q,
        hybrid_retriever=hybrid_retriever,
        llm=llm,
    )

    print("\n--- Hybrid RAG ---\n")
    print(hybrid_result["answer"])

    time.sleep(1)  # 👈 prevents rate limit

QUERY: blue patio chair cushion

--- Semantic RAG ---

Based on the provided context, I recommend the following blue patio chair cushions:

1. **LVTXIII Outdoor Seat Cushions Patio Chair Pads with Ties, All Weather Chair Cushions for Home Office and Patio Garden Furniture Decoration D16”xW17”, Stripe Beige, Set of 2** - Although this product is beige, it is a good option if you're looking for a striped design. However, I couldn't find a blue version of this product in the provided context.

2. **Indoor/Outdoor Bench Cushion, Swing Cushion, 51.2"x19.7", for Lounger Garden Furniture Patio Lounger Bench (Gray)** - This product is gray, but it's a good option if you're looking for a cushion with a cotton filling and durable fabric. It's also one of the more affordable options, priced at 29.99.

3. **Indoor/Outdoor Bench Cushion Cotton Garden Furniture Loveseat Cushion, 51.2"x19.7" Patio Wicker Seat Cushions for Lounger Garden Furniture Patio Lounger Bench (Navy Blue)** - This product is a 

In [21]:
queries_second_half = [ "something to keep the deer out of my garden without hurting them",
    "plant suitable for moving outdoors to a patio in spring",
    "best outdoor decoration that won’t fade in the sun",
    "cool gift for someone who loves gardening",
    "what is a good plant to keep outside that won’t die if I forget to water it and doesn’t need much sunlight"
]

import time

for q in queries_second_half:
    print("=" * 80)
    print(f"QUERY: {q}")
    print("=" * 80)

    semantic_result = run_semantic_rag(
        query=q,
        retriever=semantic_retriever,
        llm=llm,
        system_prompt=SYSTEM_PROMPT_FINAL,
    )

    print("\n--- Semantic RAG ---\n")
    print(semantic_result["answer"])

    time.sleep(1)  # 👈 prevents rate limit

    hybrid_result = run_hybrid_rag(
        query=q,
        hybrid_retriever=hybrid_retriever,
        llm=llm,
    )

    print("\n--- Hybrid RAG ---\n")
    print(hybrid_result["answer"])

    time.sleep(1)  # 👈 prevents rate limit

QUERY: something to keep the deer out of my garden without hurting them

--- Semantic RAG ---

Based on the provided context, here are three gift options to keep deer out of your garden without harming them:

1. **Deer Repel Deer Repellent Plants Pouches** - This product is an all-natural, long-lasting formula that repels deer and rabbits. Although the reviewer had mixed results, it's a convenient and ready-to-use solution. ($34.97)
2. **Easy Gardener 6050AZ Netting and Fencing, 7 ft X 100 ft, Black** - This product is a durable and reusable mesh deer fence that stops deer and other animals from eating plants and vegetables. Reviewers have praised its effectiveness in keeping birds and other animals out. ($24.9)
3. **Deer Out 40oz Ready-to-Use Deer Repellent** - This product is a minty fresh scent that repels deer, elk, and moose without harming them. The reviewer had great results with this product, and it's a trusted brand among professional gardeners and landscapers. ($17.95)

These

## 11. Notes for Discussion and README

Key observations so far:
- Semantic retrieval performed well on both product-specific and more abstract intent-based queries.
- Prompt design had a strong effect on grounding and usefulness.
- Including rating, average rating, and number of ratings improved recommendation quality by helping the model weigh stronger evidence more effectively.
- The final semantic RAG pipeline now works both in the notebook and in `src/rag_pipeline.py`.
- Including too many metadata fields in the hybrid RAG context caused token-limit issues, so the final context builder was simplified by removing categories and descriptions.
- The final hybrid RAG pipeline now works both in the notebook and in `src/rag_pipeline_hybrid.py`